### Data ingestion to vectordb pipeline

In [5]:
import os
from langchain_community.document_loaders import PyPDFLoader
from pathlib import Path

In [6]:

from langchain_community.document_loaders import PyPDFLoader
### Data ingestion
def pdf_processing(path):
    pdf_dir = Path(path)
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    all_docs = []
    for file in pdf_files:
        try:
            loader = PyPDFLoader(
                str(file)
            )
            documents = loader.load()
            # print('docs:', documents)
            for doc in documents:
                doc.metadata['source_file'] = file.name
                doc.metadata['file_type'] = 'pdf'
            all_docs.extend(documents)
        except Exception as e:
            print(f" Error {e}")
    return all_docs


In [7]:
print_all_pdfs = pdf_processing("../data")
print_all_pdfs

[Document(metadata={'producer': 'WeasyPrint 62.3', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Enterprise Security Architecture & Compliance Guidelines (v4.2)', 'source': '../data/pdf_files/rag_test_sample_document_IAM.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'source_file': 'rag_test_sample_document_IAM.pdf', 'file_type': 'pdf'}, page_content='STRICTLY CONFIDENTIAL // INTERNAL ONLY\nEnterprise Cloud Security Architecture &\nIdentity Access Management (IAM)\nGuidelines\nTechnical Standard for Multi-Region AWS Implementations\nDocument Reference: SEC-2026-X901\nClassification Level: Tier-1 Confidential\nEffective Date:  March 15, 2026\nOwner:  Global Infrastructure Security Group\nApproved By:  Chief Information Security Officer (CISO)\nInternal Confidential - TechCorp Global Risk & Compliance 1'),
 Document(metadata={'producer': 'WeasyPrint 62.3', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Enterprise Security Architecture & Compliance Guidelines (v4.2)', 'sourc

### Text Chunking

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
def text_splitting(documents, chunk_size=1000,chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_overlap=chunk_overlap,
        chunk_size=chunk_size,
        length_function=len,
        separators=["\n\n","\n"," ",""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f'I have split {len(documents)} documents into {len(split_docs)} chunks')
    return split_docs


In [9]:
chunks = text_splitting(print_all_pdfs)
chunks

I have split 7 documents into 15 chunks


[Document(metadata={'producer': 'WeasyPrint 62.3', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Enterprise Security Architecture & Compliance Guidelines (v4.2)', 'source': '../data/pdf_files/rag_test_sample_document_IAM.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'source_file': 'rag_test_sample_document_IAM.pdf', 'file_type': 'pdf'}, page_content='STRICTLY CONFIDENTIAL // INTERNAL ONLY\nEnterprise Cloud Security Architecture &\nIdentity Access Management (IAM)\nGuidelines\nTechnical Standard for Multi-Region AWS Implementations\nDocument Reference: SEC-2026-X901\nClassification Level: Tier-1 Confidential\nEffective Date:  March 15, 2026\nOwner:  Global Infrastructure Security Group\nApproved By:  Chief Information Security Officer (CISO)\nInternal Confidential - TechCorp Global Risk & Compliance 1'),
 Document(metadata={'producer': 'WeasyPrint 62.3', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Enterprise Security Architecture & Compliance Guidelines (v4.2)', 'sourc

### Embedding and vector store

In [10]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [11]:
class EmbeddingManager:
    def __init__(self, modelName: str = "all-MiniLM-L6-v2"):
        self.model = None
        self.model_name=modelName
        self._load_model()
    
    def _load_model(self):
        try:
            self.model = SentenceTransformer(self.model_name)
            print(f'Model loaded successfully: {self.model.get_embedding_dimension()}')
        except Exception as e:
            print(e)
            raise
    
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        if not self.model:
            raise ValueError("Model not found")
        embeddings = self.model.encode(texts, show_progress_bar = True)
        print(f'embeddings shape: {embeddings.shape}')
        return embeddings

embedding_manager = EmbeddingManager()
embedding_manager    

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5551.94it/s]


Model loaded successfully: 384


### Vectore Store

In [12]:
import chromadb

class VectorStore:
    def __init__(self, collection_name: str="pdf-documents", persistent_directory: str ="../data/vector_store"):
        self.collection_name = collection_name
        self.persistent_directory = persistent_directory
        self.client = None 
        self.collection = None
        self.initialize_store()
    
    def initialize_store(self):
        try:
            os.makedirs(self.persistent_directory, exist_ok=True)        
            self.client = chromadb.PersistentClient(path = self.persistent_directory)        
            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata={
                    "description": "PDF documents embedding for RAG"
                }
                )
            print(f"Vector store initialized: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print("Error initializing: {e}")
            raise
    
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        if len(documents)!=len(embeddings):
            raise ValueError('Length of documents and embeddings should be same')
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())

        try:
            self.collection.add(
                ids= ids,
                documents= documents_text,
                embeddings= embeddings_list,
                metadatas= metadatas,
            )        
        except Exception as e:
            print(f'Error: {e}')    

vectorStore = VectorStore()
vectorStore

Vector store initialized: pdf-documents
Existing documents in collection: 0


In [13]:
chunks[0]

Document(metadata={'producer': 'WeasyPrint 62.3', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Enterprise Security Architecture & Compliance Guidelines (v4.2)', 'source': '../data/pdf_files/rag_test_sample_document_IAM.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'source_file': 'rag_test_sample_document_IAM.pdf', 'file_type': 'pdf'}, page_content='STRICTLY CONFIDENTIAL // INTERNAL ONLY\nEnterprise Cloud Security Architecture &\nIdentity Access Management (IAM)\nGuidelines\nTechnical Standard for Multi-Region AWS Implementations\nDocument Reference: SEC-2026-X901\nClassification Level: Tier-1 Confidential\nEffective Date:  March 15, 2026\nOwner:  Global Infrastructure Security Group\nApproved By:  Chief Information Security Officer (CISO)\nInternal Confidential - TechCorp Global Risk & Compliance 1')

In [14]:
###Get the content from chunks
texts = [chunk.page_content for chunk in chunks]
texts

['STRICTLY CONFIDENTIAL // INTERNAL ONLY\nEnterprise Cloud Security Architecture &\nIdentity Access Management (IAM)\nGuidelines\nTechnical Standard for Multi-Region AWS Implementations\nDocument Reference: SEC-2026-X901\nClassification Level: Tier-1 Confidential\nEffective Date:  March 15, 2026\nOwner:  Global Infrastructure Security Group\nApproved By:  Chief Information Security Officer (CISO)\nInternal Confidential - TechCorp Global Risk & Compliance 1',
 '1. Architectural Scope & Objectives\nThis technical standard defines the mandatory security controls, identity boundary definitions, and network\nperimeter isolation strategies required for all multi-region web applications operating within TechCorp Global\ncloud infrastructure. Non-compliance with these rules automatically triggers isolated remediation workflows\nwithin the deployment orchestrator, revoking active staging routing protocols.\nCritical System Dependency\nAll parameters defined in Section 2.2 explicitly override ge

In [15]:
### Embed the text
embedded_text = embedding_manager.generate_embeddings(texts)
embedded_text

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:01<00:00,  1.26s/it]

embeddings shape: (15, 384)


array([[-0.01752929,  0.03316234, -0.00676067, ..., -0.00247252,
        -0.02593826, -0.05081594],
       [ 0.01643484,  0.03099209,  0.01665673, ..., -0.01488006,
         0.02019118, -0.10395772],
       [ 0.02995193, -0.00701952,  0.02972833, ...,  0.01444663,
        -0.00598179, -0.03093372],
       ...,
       [ 0.03453821, -0.0185665 , -0.00788263, ..., -0.05832162,
        -0.03285605,  0.02087402],
       [-0.00590307, -0.01550851, -0.01059248, ..., -0.11320703,
        -0.02943556,  0.01590506],
       [ 0.0118549 ,  0.00080957,  0.05382634, ..., -0.03982731,
         0.01627738, -0.05639936]], shape=(15, 384), dtype=float32)

In [17]:
### Store in vector DB
vectorStore.add_documents(chunks, embedded_text)

### RAG Retriever from Vector Store

In [18]:
### converts query to embedding -> hits the vectorStore -> pulls relevant data
class RAGRetriever:
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict(str, Any)]:
        print(f'Retrieving documents for the query: {query}')
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        try:
            results = self.vector_store.collection.query(
                query_embeddings = query_embedding,
                n_results = top_k
            )
            retrieved_docs = []
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorStore, embedding_manager)

In [29]:
rag_retriever.retrieve("How does compound UX-8822 achieve its safety advantage over traditional JAK1 inhibitors?")

Retrieving documents for the query: How does compound UX-8822 achieve its safety advantage over traditional JAK1 inhibitors?


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.43it/s]

embeddings shape: (1, 384)
Retrieved 2 documents (after filtering)


[{'id': 'doc_568d1d4a_9',
  'content': '1. Background & Rationale\nCompound UX-8822 is a novel, highly selective, small-molecule allosteric inhibitor of tyrosine kinase 2 (TYK2). By\nsuppressing  TYK2  activation  pathways,  the  drug  explicitly  inhibits  downstream  signaling  networks  mediated  by\ninterleukin-23 (IL-23), interleukin-12 (IL-12), and type I interferons (IFN). Crucially, at therapeutic concentrations,\nUX-8822  exhibits  more  than  50-fold  selectivity  for  TYK2  relative  to  Janus  kinase  1  (JAK1),  JAK2,  and  JAK3,\nminimizing the risk of off-target myelosuppression and lipid profile anomalies.\nAdverse Event Operational Threshold\nAny localized or systemic event demonstrating an absolute neutrophil count drop below 1,000 cells/μL dictates\nimmediate structural treatment suspension and protocol unblinding evaluation. \n2. Trial Design & Subject Stratification\nThis is a 16-week, multi-center, parallel-group study targeting an enrollment of 180 eligible subje

### Integrate retrieved vector db context with LLm output

In [20]:
### simple rag pipeline with groq pipeline
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os
load_dotenv()

### Initialise the groq llm (set api_key)
# groq_api_key = "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"
groq_api_key =  os.getenv("GROQ_API_KEY")

llm = ChatGroq(groq_api_key = groq_api_key, model_name = "llama-3.3-70b-versatile", temperature = 0.1, max_tokens=1024)

### simple rag function
def rag_simple(query, rag_retriever, llm, top_k=3):
    results = rag_retriever.retrieve(
        query,
        top_k = top_k
    )
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question"
    prompt = """Use the following context to answer the question precisely.
        Context:
        {context}
        
        Question:
        {query}

        Answer:"""
    response_llm = llm.invoke([prompt.format(context=context,query=query)])
    return response_llm.content

In [27]:
answer = rag_simple("What are the entry criteria thresholds for a patient regarding their PASI score and static Physician's Global Assessment rating?", rag_retriever, llm)
print(answer)

Retrieving documents for the query: What are the entry criteria thresholds for a patient regarding their PASI score and static Physician's Global Assessment rating?


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.44it/s]


embeddings shape: (1, 384)
Retrieved 2 documents (after filtering)
The entry criteria thresholds for a patient are: 
1. PASI Score ≥ 12.0 
2. static Physician's Global Assessment (sPGA) rating of Grade ≥ 3 (Moderate).


### Enhanced RAG pipeline

In [28]:
def enhanced_rag_pipeline(query, rag_retriever, llm, top_k=5, min_score = 0.2, return_context = False):
    results = rag_retriever.retrieve(
        query,
        top_k = top_k,
        score_threshold = min_score
    )
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    prompt = """Use the following context to answer the question precisely.
        Context:
        {context}
        
        Question:
        {query}

        Answer:"""
    response = llm.invoke([prompt.format(context = context, query = query)])
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output
result = enhanced_rag_pipeline("How does compound UX-8822 achieve its safety advantage over traditional JAK1 inhibitors?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)

Retrieving documents for the query: How does compound UX-8822 achieve its safety advantage over traditional JAK1 inhibitors?


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.58it/s]

embeddings shape: (1, 384)
Retrieved 0 documents (after filtering)


In [25]:
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])


Answer: No relevant context found.
Sources: []
Confidence: 0.0
Context Preview: 
